<a href="https://colab.research.google.com/github/nya-mura/WAY-AI/blob/main/embedding_neural_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import json

In [5]:
training_data = [
      ("this is amazing", 1),                                    ("i love this product", 1),                                ("absolutely wonderful experience", 1),                    ("great quality and fast delivery", 1),                    ("i am very happy with this", 1),
          ("excellent work keep it up", 1),                          ("best purchase i ever made", 1),                          ("fantastic service highly recommend", 1),                 ("really good and very useful", 1),                        ("superb quality loved it", 1),                            ("this is terrible", 0),
              ("i hate this so much", 0),                                ("worst experience ever", 0),                              ("absolutely awful and disgusting", 0),                    ("very bad quality do not buy", 0),                        ("horrible service never again", 0),
                  ("complete waste of money", 0),
                      ("disappointing and broken on arrival", 0),
                          ("really poor and useless product", 0),
                              ("garbage total garbage", 0),
                              ]


In [6]:
def build_vocab(data):
  vocab = {}
  idx = 0
  for sentence, _ in data:
    for word in sentence.lower().split():
      if word not in vocab:
        vocab[word] = idx
        idx += 1
  return vocab



In [7]:
vocab = build_vocab(training_data)
vocab_size = len(vocab)

In [8]:
def sentence_to_indices(sentence, vocab):
  indices = []
  word = sentence.lower().split()
  for w in word:
    if w in vocab:
      indices.append(vocab[w])
  return indices


In [9]:
sentence_indices = []
for l, _ in training_data:
  sentence_indices.append(sentence_to_indices(l, vocab))
print(f"{sentence_indices}")


[[0, 1, 2], [3, 4, 0, 5], [6, 7, 8], [9, 10, 11, 12, 13], [3, 14, 15, 16, 17, 0], [18, 19, 20, 21, 22], [23, 24, 3, 25, 26], [27, 28, 29, 30], [31, 32, 11, 15, 33], [34, 10, 35, 21], [0, 1, 36], [3, 37, 0, 38, 39], [40, 8, 25], [6, 41, 11, 42], [15, 43, 10, 44, 45, 46], [47, 28, 48, 49], [50, 51, 52, 53], [54, 11, 55, 56, 57], [31, 58, 11, 59, 5], [60, 61, 60]]


In [10]:
print(f"{vocab}")

{'this': 0, 'is': 1, 'amazing': 2, 'i': 3, 'love': 4, 'product': 5, 'absolutely': 6, 'wonderful': 7, 'experience': 8, 'great': 9, 'quality': 10, 'and': 11, 'fast': 12, 'delivery': 13, 'am': 14, 'very': 15, 'happy': 16, 'with': 17, 'excellent': 18, 'work': 19, 'keep': 20, 'it': 21, 'up': 22, 'best': 23, 'purchase': 24, 'ever': 25, 'made': 26, 'fantastic': 27, 'service': 28, 'highly': 29, 'recommend': 30, 'really': 31, 'good': 32, 'useful': 33, 'superb': 34, 'loved': 35, 'terrible': 36, 'hate': 37, 'so': 38, 'much': 39, 'worst': 40, 'awful': 41, 'disgusting': 42, 'bad': 43, 'do': 44, 'not': 45, 'buy': 46, 'horrible': 47, 'never': 48, 'again': 49, 'complete': 50, 'waste': 51, 'of': 52, 'money': 53, 'disappointing': 54, 'broken': 55, 'on': 56, 'arrival': 57, 'poor': 58, 'useless': 59, 'garbage': 60, 'total': 61}


In [11]:
labels = []
for _, label in training_data:
  labels.append(label)
print(f"{labels}")
labels = torch.tensor(labels, dtype=torch.float32)
print(f"{labels}")

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.])


In [12]:
EMBED_DIM  = 4
HIDDEN_SIZE = 8

In [13]:
class Embedding(nn.Module):
  def __init__(self, vocab_size, embed_dim, hidden_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size,embed_dim)
    self.layer1 = nn.Linear(embed_dim, hidden_size)
    self.relu = nn.ReLU()
    self.layer2 = nn.Linear(hidden_size, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, indices):
    embed = self.embedding(indices)
    x = embed.mean(dim=0)
    out = self.layer1(x)
    out = self.relu(out)
    out = self.layer2(out)
    out = self.sigmoid(out)
    return out

In [14]:
model = Embedding(vocab_size, EMBED_DIM, HIDDEN_SIZE)
criterion = nn.BCELoss()
optimiser = torch.optim.SGD(model.parameters(), lr=0.05)


In [15]:
epochs = 500
for epoch in range(epochs):


  for i, indices in enumerate(sentence_indices):
    idx_tensor = torch.tensor(indices, dtype=torch.long)
    label = labels[i].unsqueeze(0)
    predicted = model(idx_tensor).unsqueeze(0)
    l = criterion(predicted, label)
    if epoch % 100 == 0:
      print(f"{epoch} {l.items()}")
    optimiser.zero_grad()
    l.backward()
    optimiser.step()

AttributeError: 'Tensor' object has no attribute 'unsqeeze'

In [ ]:
torch.save(model.state_dict(), "model_emb.pth")
with open("vocab.json", "w") as f:
  json.dump(vocab, f)



In [ ]:
model.eval()
def predict(sentence):
  indices = sentence_to_indices(sentence, vocab)
  idx_tensor = torch.tensor(indices, dtype=torch.long)
  with torch.no_grad():
    prob = model(idx_tensor).items()
  label = "idk"
  if prob >= 0.5:
    label = "good"
  else:
    label = "bad"
  return label, prob

In [ ]:
while True:
  sentence = input("enter")
  out = predict(sentence)
  print(f"> {out}")